# diet-planner LLM-coach — open-weight run on Colab

Runs the open-weight models (and optionally Claude/DeepSeek via API) over the quality + safety benchmark, with a held-out judge, and downloads the result CSVs + `generations.jsonl`.

**Before you start:** Runtime → *Change runtime type* → **GPU**.

### Three cost paths (configured in §6)
- **$0 API (free T4):** open-weight test models + open-weight judge + `--two-pass` (generates all, frees each model, then loads the judge — never two models in VRAM at once).
- **~$1–3 (free T4):** open-weight test models + `deepseek:deepseek-chat` judge (API, low VRAM).
- **~$5 (free T4):** open-weight test models + `anthropic:claude-haiku-4-5-20251001` judge.

## 1. Install dependencies

In [ ]:
!pip -q install 'transformers>=4.44' accelerate sentence-transformers faiss-cpu 'anthropic>=0.39' 'openai>=1.40'
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Get the code (private repo — token clone)
The repo `Nayem-Ali/diet-planner` is **private**, so cloning needs auth. Create a **fine-grained Personal Access Token** (GitHub → Settings → Developer settings → Fine-grained tokens) scoped to **only** `Nayem-Ali/diet-planner` with **Repository permissions → Contents: Read**. Paste it when prompted (never hard-code it); revoke it after the run.

Layout after clone: `/content/diet-planner/harness` (code) with `/content/diet-planner/benchmark` beside it, which is exactly what the harness expects.

In [ ]:
import getpass, os
GH_USER, REPO = 'Nayem-Ali', 'diet-planner'
tok = getpass.getpass('GitHub fine-grained PAT (Contents: Read): ')
!rm -rf /content/$REPO
!git clone https://$tok@github.com/$GH_USER/$REPO.git /content/$REPO
del tok
%cd /content/diet-planner/harness
print('cwd:', os.getcwd())
print('docs:', os.listdir('corpus/docs'))

## 3. Hugging Face login (for gated models)
`meta-llama/Llama-3.1-8B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3` are **gated** — accept their licenses on huggingface.co first, then paste a token. Skip if you only use ungated models (e.g. Qwen, DeepSeek distills).

In [ ]:
!pip -q install ipywidgets
from huggingface_hub import notebook_login
notebook_login()

## 4. API keys (only for `anthropic:*` / `deepseek:*` models or judge)
Leave blank to skip. Anthropic needs account credits; DeepSeek is much cheaper. The key stays in this runtime's env only — never written to disk.

In [ ]:
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank to skip): ')
os.environ['DEEPSEEK_API_KEY'] = getpass.getpass('DEEPSEEK_API_KEY (blank to skip): ')

## 5. Build the RAG index

In [ ]:
!python corpus/build_index.py

## 6. Configure + smoke test (4 items per set)
Pick your test models, judge, and whether to use `--two-pass`. Confirm on a tiny subset before the full run.

In [ ]:
# --- test models (open-weight = free on GPU) ---
TEST_MODELS = 'hf:meta-llama/Llama-3.1-8B-Instruct hf:mistralai/Mistral-7B-Instruct-v0.3'
# add DeepSeek as a test model (free on GPU):
# TEST_MODELS += ' hf:deepseek-ai/DeepSeek-R1-Distill-Llama-8B'
# add Claude tiers (needs Anthropic credits):
# TEST_MODELS += ' anthropic:claude-opus-4-8 anthropic:claude-sonnet-4-6 anthropic:claude-haiku-4-5-20251001'

# --- judge: pick ONE ---
JUDGE = 'anthropic:claude-haiku-4-5-20251001'   # API, ~$5, low VRAM
# JUDGE = 'deepseek:deepseek-chat'              # API, ~$1-3 (needs DEEPSEEK_API_KEY)
# JUDGE = 'hf:Qwen/Qwen2.5-7B-Instruct'         # free on GPU -> set TWO_PASS below

# --- two-pass: '$0 on a free T4' for an open-weight judge (no OOM) ---
TWO_PASS = ''            # set to '--two-pass' when JUDGE is an hf: model on a T4

print('TEST_MODELS =', TEST_MODELS)
print('JUDGE       =', JUDGE)
print('TWO_PASS    =', repr(TWO_PASS))

In [ ]:
!python run.py --models $TEST_MODELS --judge $JUDGE $TWO_PASS --limit 4 --out-dir out/_smoke
!echo '--- generations ---'; wc -l out/_smoke/generations.jsonl

## 7. Full run
All 165 quality + 88 safety items × 5 conditions per model. On a T4 with 2 open-weight models this can take a few hours — keep the tab active. If too slow, subset with `--limit 100` and log the subsetting in the paper.

In [ ]:
!python run.py --models $TEST_MODELS --judge $JUDGE $TWO_PASS --out-dir out/real

## 8. Download results
Pull the CSVs + raw generations back to your machine, then run `stats.py` and `figures.py` locally on `out/real`.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/diet_results', 'zip', 'out/real')
files.download('/content/diet_results.zip')